## 3. Create a knowledge source

A knowledge source is a reusable reference to source data. The following code creates a knowledge source that targets the index we previously created.


In [11]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference, SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    KnowledgeBase, KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode, KnowledgeSourceReference
)
from azure.identity import DefaultAzureCredential
import os

SEARCH_ENDPOINT = os.environ["SEARCH_ENDPOINT"]
INDEX_MANUALS = "product-manuals"
INDEX_CATALOG = "product-catalog"

knowledge_source_name_manuals = INDEX_MANUALS + "-knowledge-source"
knowledge_source_name_catalog = INDEX_CATALOG + "-knowledge-source"

base_name = "product-info"

credential = DefaultAzureCredential()

In [8]:
ks_manuals = SearchIndexKnowledgeSource(
    name=knowledge_source_name_manuals,
    description="Knowledge source for product manuals",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_MANUALS,
        source_data_fields=[
            SearchIndexFieldReference(name="title"), 
            SearchIndexFieldReference(name="page"),
            SearchIndexFieldReference(name="blob_url")
        ]
    ),
)

ks_catalog = SearchIndexKnowledgeSource(
    name=knowledge_source_name_catalog,
    description="Knowledge source for product catalog",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_CATALOG,
        source_data_fields=[
            SearchIndexFieldReference(name="ProductName"), 
            SearchIndexFieldReference(name="ProductID")
        ]
    ),
)

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)

index_client.create_or_update_knowledge_source(knowledge_source=ks_manuals)
print(f"Knowledge source '{knowledge_source_name_manuals}' created or updated successfully.")

index_client.create_or_update_knowledge_source(knowledge_source=ks_catalog)
print(f"Knowledge source '{knowledge_source_name_catalog}' created or updated successfully.")

Knowledge source 'product-manuals-knowledge-source' created or updated successfully.
Knowledge source 'product-catalog-knowledge-source' created or updated successfully.


### 3.1 Create a knowledge base

The following code creates a knowledge base that orchestrates agentic retrieval from the knowledge sources above.

In [12]:
knowledge_base = KnowledgeBase(
    name=base_name,
    knowledge_sources=[
        KnowledgeSourceReference(
            name=knowledge_source_name_manuals
        ),
        KnowledgeSourceReference(
            name=knowledge_source_name_catalog
        )
    ],
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort()
)


index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
index_client.create_or_update_knowledge_base(knowledge_base=knowledge_base)

print(f"Knowledge base '{base_name}' created or updated successfully")

Knowledge base 'product-info' created or updated successfully
